# 第17章：Tensor Core 深入 -- tl.dot 的秘密

## 前置知识
- 第09章：分块矩阵乘法基础
- 第16章：混合精度策略
- 了解 GPU warp (32线程) 的概念

## 学习目标
- 理解 **Tensor Core** 是什么 (硬件矩阵乘法单元)
- 理解 `tl.dot` 如何映射到 `mma.sync` PTX 指令
- 理解 CUDA WMMA API 和 MMA PTX 的区别
- 理解为什么 Block Size 必须是 16 的倍数
- 对比 CUDA 手写 Tensor Core 代码 vs Triton 的简洁性

## 对应 CUDA 代码
- `src/wmma/00wmma_naive.cu` — WMMA API (高层封装, 1.734ms)
- `src/mma/00mma_naive.cu` — MMA PTX 内联汇编 (2.445ms, naive)
- `src/mma/04mma_pipline.cu` — MMA PTX + 流水线 (0.411ms, 优化版)

In [ ]:
import torch
import triton
import triton.language as tl

## 17.1 Tensor Core 是什么？

### 硬件概述

Tensor Core 是 NVIDIA GPU 中的专用**矩阵乘法硬件单元**，
与传统的 CUDA Core (标量 FMA) 完全不同：

```
CUDA Core (标量运算):              Tensor Core (矩阵运算):
  一次计算: a * b + c              一次计算: D = A * B + C
  操作数: 标量                      操作数: 矩阵片段
  FP32: 1 FLOP/cycle/core         FP16: 几十~几百 FLOP/cycle/core
  
  ┌─┐   ┌─┐   ┌─┐                ┌─────┐   ┌─────┐   ┌─────┐
  │a│ * │b│ + │c│ = d             │  A  │ * │  B  │ + │  C  │ = D
  └─┘   └─┘   └─┘                │m×k  │   │k×n  │   │m×n  │   m×n
                                  └─────┘   └─────┘   └─────┘
```

### Tensor Core 的矩阵大小

不同架构支持的原子矩阵乘大小：

```
架构       指令                  矩阵大小 (M x N x K)     精度
────────────────────────────────────────────────────────────────
Volta      WMMA                  16 x 16 x 16             FP16→FP16/FP32
Turing     WMMA/MMA              16 x 16 x 16 / 16x8x8   FP16→FP16/FP32
Ampere     MMA                   16 x 8 x 8               FP16/BF16/TF32
                                 16 x 8 x 16              FP16/BF16
Hopper     WGMMA                 64 x N x 16              各种精度
```

### Warp 级别操作

Tensor Core 是 **warp 级别** 的操作 —— 一个 warp (32 线程) 协作完成一次矩阵乘：

```
WMMA (16x16x16):
                                    
  A (16x16)     B (16x16)     C (16x16)
  ┌─────────┐   ┌─────────┐   ┌─────────┐
  │ Thread 0│   │ Thread 0│   │ Thread 0│
  │ Thread 1│   │ Thread 1│   │ Thread 1│
  │   ...   │ @ │   ...   │ = │   ...   │
  │Thread 31│   │Thread 31│   │Thread 31│
  └─────────┘   └─────────┘   └─────────┘
  
  每个线程持有矩阵的几个元素 (fragment)
  32 个线程的 fragment 合在一起构成完整矩阵
  硬件一条指令完成整个矩阵乘!
```

## 17.2 CUDA 的三种 Tensor Core 编程方式

### 方式 1: WMMA API (高层封装)

```cpp
// 来自 wmma_naive.cu
#include <mma.h>
using namespace nvcuda::wmma;

// 1. 声明 fragment (矩阵片段)
fragment<matrix_a, 16, 16, 16, half, row_major> a_frag;
fragment<matrix_b, 16, 16, 16, half, row_major> b_frag;
fragment<accumulator, 16, 16, 16, half> c_frag;

// 2. 初始化累加器
fill_fragment(c_frag, half(0.0f));

// 3. 加载 + 计算
for (int k = 0; k < K; k += 16) {
    load_matrix_sync(a_frag, A_ptr + offset, ldA);
    load_matrix_sync(b_frag, B_ptr + offset, ldB);
    mma_sync(c_frag, a_frag, b_frag, c_frag);  // D = A * B + C
}

// 4. 存储
store_matrix_sync(C_ptr + offset, c_frag, ldC, mem_row_major);
```

**特点**：相对简单, 但性能不如手写 MMA, 灵活性有限。

### 方式 2: MMA PTX 内联汇编 (底层控制)

```cpp
// 来自 mma_naive.cu  
// 寄存器声明
uint32_t regD[2] = {0, 0};  // 输出 fragment

// PTX 内联汇编: mma.sync.aligned.m16n8k8
asm volatile(
    "mma.sync.aligned.m16n8k8.row.col.f16.f16.f16.f16 "
    "{%0, %1}, "         // D: 2个 u32 寄存器
    "{%2, %3}, "         // A: 2个 u32 寄存器
    "{%4}, "             // B: 1个 u32 寄存器
    "{%5, %6}; "         // C: 2个 u32 寄存器 (累加器)
    :"=r"(regD[0]), "=r"(regD[1])
    :"r"(*(uint32_t*)(A_ptr + thread_offset_a)),
     "r"(*(uint32_t*)(A_ptr + thread_offset_a + 8*K)),
     "r"(*(uint32_t*)(B_ptr + thread_offset_b)),
     "r"(regD[0]), "r"(regD[1])
);
```

**特点**：最大灵活性, 最高性能潜力, 但代码极其复杂。

### 方式 3: Triton tl.dot (自动映射)

```python
# 整个 Tensor Core 调用, 1 行 Python!
acc = tl.dot(a_tile, b_tile, acc=acc)
```

**特点**：编译器自动映射到最优的 MMA 指令, 开发效率极高。

## 17.3 tl.dot 如何映射到 mma.sync

### 编译器的工作

当 Triton 编译器看到 `tl.dot(a, b, acc=acc)` (假设 a 是 128x32, b 是 32x128) 时：

```
Step 1: 确定可用的 MMA 指令
  GPU = Ampere → mma.sync.aligned.m16n8k16.row.col.f16.f16.f16.f16
  原子操作大小: (16, 8, 16)

Step 2: 将大矩阵分解为 MMA 操作
  a (128x32) @ b (32x128) 需要:
  M方向: 128/16 = 8 个 MMA tiles
  N方向: 128/8  = 16 个 MMA tiles
  K方向: 32/16  = 2 步 K 迭代
  总计: 8 * 16 * 2 = 256 个 mma.sync 指令

Step 3: 将 MMA 操作分配给 warp
  每个 warp 处理若干 MMA tile
  warp 内的 32 个线程协作完成每个 MMA

Step 4: 安排数据搬运
  使用 ldmatrix 从 shared memory 加载到 MMA 寄存器
  确保数据对齐和布局正确
```

### 数据在线程间的分布

```
mma.sync.m16n8k8 的线程分布:

A 矩阵 (16x8, row-major):          B 矩阵 (8x8, col-major):
行: 0-7   → Thread 0-3 的 reg A    行: 0-7  → Thread 0-3 的 reg B
行: 8-15  → Thread 0-3 的 reg A'   (Lane 0: B[0:2], Lane 1: B[2:4]...)

每个线程持有:
  A: 2个 u32 寄存器 (= 4个 FP16 元素)
  B: 1个 u32 寄存器 (= 2个 FP16 元素)
  D: 2个 u32 寄存器 (= 4个 FP16 元素)

线程到矩阵元素的映射 (mma.m16n8k8):
  Thread(lane_id) 持有的元素:
  D 矩阵行: lane_id / 4         (行 0-7)
  D 矩阵列: (lane_id % 4) * 2   (列 0,2,4,6)
  regD[0]: D[row, col] 和 D[row, col+1]
  regD[1]: D[row+8, col] 和 D[row+8, col+1]
```

**Triton 对开发者隐藏了所有这些细节！**

## 17.4 Block Size 要求

### 为什么必须是 16 的倍数？

```
Tensor Core 的原子操作是 16xNxK 的矩阵乘:
  WMMA: 16x16x16
  MMA:  16x8x8 或 16x8x16

因此:
  BLOCK_M 必须是 16 的倍数 (M 方向按 16 划分)
  BLOCK_N 必须是 16 的倍数 (最好是 8 的倍数, 对 mma.m16n8k8)
  BLOCK_K 必须是 16 的倍数 (K 方向按 16 或 8 划分)
```

### 不满足要求时会怎样？

```python
# BLOCK_M=48 → 48 / 16 = 3 → OK (是16的倍数? 48=16*3, OK!)
# BLOCK_M=50 → 不是 16 的倍数 → Triton 编译器报错或退化到非 TC 路径
# BLOCK_K=24 → 不是 16 的倍数 → 可能报错
```

### 常见的有效 Block Size 组合

```
BLOCK_M  BLOCK_N  BLOCK_K  | 说明
──────────────────────────────────────
  16       16       16     | 最小有效 TC 配置
  32       32       16     | 小型 kernel
  64       64       32     | 中等
  128      128      32     | 常用最佳配置
  128      256      32     | 高算术强度
  256      128      32     | 高算术强度
  256      256      64     | 大配置 (需要大量 smem)
```

In [ ]:
# ========== Tensor Core GEMM kernel ==========
@triton.jit
def matmul_tc_kernel(
    a_ptr, b_ptr, c_ptr,
    M, N, K,
    stride_am, stride_ak,
    stride_bk, stride_bn,
    stride_cm, stride_cn,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
    GROUP_SIZE_M: tl.constexpr,
):
    """
    标准 GEMM kernel, 使用 Tensor Core (通过 tl.dot)。
    
    一行 tl.dot 等价于 CUDA 中:
    - WMMA: ~15 行 (fragment 声明 + load + mma_sync + store)
    - MMA PTX: ~30 行 (寄存器声明 + asm + 手动存储)
    - MMA + ldmatrix: ~50 行 (加上 smem 布局优化)
    """
    pid = tl.program_id(0)
    grid_m = tl.cdiv(M, BLOCK_M)
    grid_n = tl.cdiv(N, BLOCK_N)
    group_id = pid // (GROUP_SIZE_M * grid_n)
    first_pid_m = group_id * GROUP_SIZE_M
    group_size_m = min(grid_m - first_pid_m, GROUP_SIZE_M)
    pid_m = first_pid_m + ((pid % (GROUP_SIZE_M * grid_n)) % group_size_m)
    pid_n = (pid % (GROUP_SIZE_M * grid_n)) // group_size_m
    
    a_block_ptr = tl.make_block_ptr(
        base=a_ptr, shape=(M, K), strides=(stride_am, stride_ak),
        offsets=(pid_m * BLOCK_M, 0), block_shape=(BLOCK_M, BLOCK_K), order=(1, 0),
    )
    b_block_ptr = tl.make_block_ptr(
        base=b_ptr, shape=(K, N), strides=(stride_bk, stride_bn),
        offsets=(0, pid_n * BLOCK_N), block_shape=(BLOCK_K, BLOCK_N), order=(1, 0),
    )
    
    # FP32 累加器
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    
    for k in range(0, K, BLOCK_K):
        a_tile = tl.load(a_block_ptr, boundary_check=(0, 1))
        b_tile = tl.load(b_block_ptr, boundary_check=(0, 1))
        # 这一行在编译后变成:
        # - ldmatrix 指令 (从 smem 加载到 MMA 寄存器)
        # - mma.sync.aligned 指令 (Tensor Core 矩阵乘)
        # - 可能的多轮 ldmatrix + mma (取决于 BLOCK_M/N/K vs MMA 原子大小)
        acc = tl.dot(a_tile, b_tile, acc=acc)
        a_block_ptr = tl.advance(a_block_ptr, (0, BLOCK_K))
        b_block_ptr = tl.advance(b_block_ptr, (BLOCK_K, 0))
    
    c_block_ptr = tl.make_block_ptr(
        base=c_ptr, shape=(M, N), strides=(stride_cm, stride_cn),
        offsets=(pid_m * BLOCK_M, pid_n * BLOCK_N),
        block_shape=(BLOCK_M, BLOCK_N), order=(1, 0),
    )
    tl.store(c_block_ptr, acc.to(tl.float16), boundary_check=(0, 1))


def matmul_tc(a, b, BLOCK_M=128, BLOCK_N=128, BLOCK_K=32, GROUP_SIZE_M=8):
    M, K = a.shape
    K2, N = b.shape
    assert K == K2
    c = torch.empty((M, N), device=a.device, dtype=a.dtype)
    grid = (triton.cdiv(M, BLOCK_M) * triton.cdiv(N, BLOCK_N),)
    matmul_tc_kernel[grid](
        a, b, c, M, N, K,
        a.stride(0), a.stride(1), b.stride(0), b.stride(1),
        c.stride(0), c.stride(1),
        BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, BLOCK_K=BLOCK_K,
        GROUP_SIZE_M=GROUP_SIZE_M,
    )
    return c

In [ ]:
# ========== 正确性验证 ==========
torch.manual_seed(42)
M, N, K = 2048, 2048, 1024
a = torch.randn(M, K, device='cuda', dtype=torch.float16)
b = torch.randn(K, N, device='cuda', dtype=torch.float16)

c_tc = matmul_tc(a, b)
c_ref = torch.matmul(a, b)

print(f"Tensor Core GEMM 正确性:")
print(f"  最大误差: {(c_tc - c_ref).abs().max().item():.4f}")
print(f"  相对误差: {torch.norm(c_tc.float()-c_ref.float()) / torch.norm(c_ref.float()):.6f}")
print(f"  通过: {torch.allclose(c_tc, c_ref, atol=1.0)}")

## 17.5 CUDA WMMA vs MMA PTX vs Triton 对比表

### 代码量对比

| 方面 | WMMA API | MMA PTX | Triton tl.dot |
|------|---------|---------|---------------|
| **Fragment 声明** | 3行 (a,b,c fragment) | ~5行 (uint32_t 数组) | 0行 |
| **数据加载** | load_matrix_sync (1行) | 手动地址计算+ldmatrix (5-10行) | tl.load (1行) |
| **矩阵乘** | mma_sync (1行) | asm volatile mma.sync (~10行) | tl.dot (1行) |
| **结果存储** | store_matrix_sync (1行) | 手动地址计算+写入 (5行) | tl.store (1行) |
| **线程映射** | 自动 | 手动 (lane_id/4, lane_id%4) | 自动 |
| **Smem 管理** | 不需要 (直接全局内存) | 手动 (可选, 但性能关键) | 自动 |
| **Pipeline** | 手动 | 手动 (双缓冲, ~100行) | num_stages 参数 |
| **总代码行数** | ~50行 | ~150-250行 | ~20行 |
| **性能** | 中等 | 最高 (如果写得好) | 接近最高 |
| **开发时间** | 小时 | 天~周 | 分钟 |

### 核心代码对比

#### CUDA WMMA (wmma_naive.cu):
```cpp
// 声明 fragments (~3行)
fragment<matrix_a, 16, 16, 16, half, row_major> a_frag;
fragment<matrix_b, 16, 16, 16, half, row_major> b_frag;
fragment<accumulator, 16, 16, 16, half> c_frag;
fill_fragment(c_frag, half(0.0f));

// K 循环 (~4行)
for (int k = 0; k < K; k += WMMA_K) {
    load_matrix_sync(a_frag, blockA_ptr + offset, K);
    load_matrix_sync(b_frag, blockB_ptr + offset, N);
    mma_sync(c_frag, a_frag, b_frag, c_frag);
}
store_matrix_sync(blockC_ptr + offset, c_frag, N, mem_row_major);
```

#### CUDA MMA PTX (mma_naive.cu):
```cpp
// 手动线程映射
int offset_thread_cx = (lane_id % 4) * 2;
int offset_thread_cy = lane_id / 4;
uint32_t regD[2] = {0, 0};

// K 循环 + 内联汇编 (~15行)
for (int k = 0; k < K; k += MMA_K) {
    asm volatile(
        "mma.sync.aligned.m16n8k8.row.col.f16.f16.f16.f16 "
        "{%0, %1}, {%2, %3}, {%4}, {%5, %6}; "
        :"=r"(regD[0]), "=r"(regD[1])
        :"r"(*(uint32_t*)(warpA_ptr + cy*K + k + cx)),
         "r"(*(uint32_t*)(warpA_ptr + (cy+8)*K + k + cx)),
         "r"(*(uint32_t*)(warpB_ptr + cy*K + k + cx)),
         "r"(regD[0]), "r"(regD[1])
    );
}
// 手动写回
*(uint32_t*)(warpC_ptr + cy*N + cx) = regD[0];
*(uint32_t*)(warpC_ptr + (cy+8)*N + cx) = regD[1];
```

#### Triton:
```python
# 全部! (~5行)
acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
for k in range(0, K, BLOCK_K):
    a = tl.load(a_block_ptr, boundary_check=(0, 1))
    b = tl.load(b_block_ptr, boundary_check=(0, 1))
    acc = tl.dot(a, b, acc=acc)
```

In [ ]:
# ========== Block Size 对 Tensor Core 利用率的影响 ==========
M, N, K = 2048, 2048, 1024

def benchmark_tc(M, N, K, BM, BN, BK, num_warmup=10, num_rep=50):
    a = torch.randn(M, K, device='cuda', dtype=torch.float16)
    b = torch.randn(K, N, device='cuda', dtype=torch.float16)
    
    for _ in range(num_warmup):
        matmul_tc(a, b, BLOCK_M=BM, BLOCK_N=BN, BLOCK_K=BK)
    torch.cuda.synchronize()
    
    s = torch.cuda.Event(enable_timing=True)
    e = torch.cuda.Event(enable_timing=True)
    s.record()
    for _ in range(num_rep):
        matmul_tc(a, b, BLOCK_M=BM, BLOCK_N=BN, BLOCK_K=BK)
    e.record()
    torch.cuda.synchronize()
    ms = s.elapsed_time(e) / num_rep
    tflops = 2.0 * M * N * K / (ms * 1e-3) / 1e12
    return ms, tflops

print(f"Block Size 对 Tensor Core GEMM 的影响 (M={M}, N={N}, K={K})")
print(f"{'BM':>6} {'BN':>6} {'BK':>6} | {'MMA tiles':>12} | {'时间(ms)':>10} {'TFLOPS':>8}")
print("-" * 60)

for BM, BN, BK in [
    (16, 16, 16),
    (32, 32, 16),
    (64, 64, 16),
    (64, 64, 32),
    (128, 128, 32),
    (128, 128, 64),
    (128, 256, 32),
    (256, 128, 32),
]:
    # 估算 MMA tile 数 (假设 m16n8k16)
    mma_m = BM // 16
    mma_n = BN // 8
    mma_k = BK // 16
    mma_tiles = mma_m * mma_n * mma_k
    try:
        ms, tflops = benchmark_tc(M, N, K, BM, BN, BK)
        print(f"{BM:>6} {BN:>6} {BK:>6} | {mma_tiles:>12} | {ms:>10.3f} {tflops:>8.2f}")
    except Exception as e:
        print(f"{BM:>6} {BN:>6} {BK:>6} | {mma_tiles:>12} | ERROR: {str(e)[:30]}")

## 17.6 Layout 要求与对齐

### Tensor Core 对数据布局的要求

```
MMA 指令对输入矩阵有严格的布局要求:

mma.sync.m16n8k8.row.col:
  A: row-major (行主序)
  B: col-major (列主序)  ← 注意!

这就是为什么 CUDA mma_naive.cu 中 B 矩阵需要转置:
  B.transpose();  // Host 端预处理
  blockB_ptr = B + blockIdx.x * BlockTileN * K;  // B 按列存储
  
在 Triton 中:
  编译器自动处理布局转换!
  你可以传入行主序的 B, 编译器会在 smem 中重新排布
```

### 对齐要求

```
Tensor Core 要求数据地址对齐:
  FP16: 16-byte alignment (8 个 FP16 元素)
  BF16: 16-byte alignment
  
Triton 自动保证对齐:
  - make_block_ptr 确保 tile 对齐
  - 编译器在 smem 中安排对齐的布局

CUDA 中需要手动保证:
  - 矩阵的 leading dimension 必须是 8 的倍数
  - smem 的 base address 自动对齐
  - 但 offset 计算必须保持对齐
```

In [ ]:
# ========== Triton vs cuBLAS 在 Tensor Core 级别的对比 ==========
print("Triton Tensor Core GEMM vs cuBLAS")
print(f"{'M':>6} {'N':>6} {'K':>6} | {'Triton(ms)':>12} {'cuBLAS(ms)':>12} | {'Triton/cuBLAS':>14}")
print("-" * 70)

for M, N, K in [
    (512, 512, 512),
    (1024, 1024, 1024),
    (2048, 2048, 1024),
    (2048, 2048, 2048),
    (4096, 4096, 2048),
    (4096, 4096, 4096),
]:
    a = torch.randn(M, K, device='cuda', dtype=torch.float16)
    b = torch.randn(K, N, device='cuda', dtype=torch.float16)
    
    # Triton
    ms_tri, _ = benchmark_tc(M, N, K, 128, 128, 32)
    
    # cuBLAS
    for _ in range(10):
        torch.matmul(a, b)
    torch.cuda.synchronize()
    s, e = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
    s.record()
    for _ in range(50):
        torch.matmul(a, b)
    e.record()
    torch.cuda.synchronize()
    ms_cu = s.elapsed_time(e) / 50
    
    ratio = ms_tri / ms_cu
    print(f"{M:>6} {N:>6} {K:>6} | {ms_tri:>11.3f}ms {ms_cu:>11.3f}ms | {ratio:>13.2f}x")

## 17.7 总结

### 本章要点

1. **Tensor Core 硬件**：GPU 中的专用矩阵乘法单元，一条指令完成小矩阵乘法
   - WMMA: 16x16x16, MMA: 16x8x8 等
   - Warp 级别操作 (32 线程协作)

2. **三种编程方式**：
   | 方式 | 代码量 | 灵活性 | 性能 |
   |------|--------|--------|------|
   | WMMA API | ~50行 | 中 | 中 |
   | MMA PTX | ~200行 | 最高 | 最高 |
   | Triton tl.dot | ~5行 | 中高 | 接近最高 |

3. **tl.dot 的工作原理**：
   - 编译器根据 block shape 和 GPU 架构选择最优的 MMA 指令
   - 自动处理数据布局、对齐、寄存器分配
   - 自动使用 ldmatrix 高效加载

4. **Block Size 要求**：
   - BLOCK_M, BLOCK_N: 必须是 16 的倍数
   - BLOCK_K: 必须是 16 的倍数
   - 更大的 block → 更好的数据复用，但受 smem 限制

5. **100 行 CUDA = 1 行 Triton**：
   - 100+ 行的 CUDA MMA 代码 (寄存器声明 + 地址计算 + asm + 存储) 在 Triton 中只需 1 行 `tl.dot`
   - 编译器承担了所有底层复杂性

### 练习

1. **PTX 查看**：使用 `kernel.warmup(...)` 后查看 PTX，找到 `mma.sync` 指令
2. **非16对齐**：尝试 BLOCK_M=96, BLOCK_N=96，观察编译器行为
3. **性能分析**：使用 NCU profiler 查看 Tensor Core 利用率
4. **思考题**：为什么 WMMA API (wmma_naive.cu, 1.734ms) 比 MMA naive (mma_naive.cu, 2.445ms) 更快？
   （提示：WMMA 的内部实现可能做了 smem 优化）

### 下一章预告

第18章将是 **终极优化 GEMM** —— 组合所有技巧 (block_ptr, swizzle, pipeline, precision, autotuning)，
实现一个接近 cuBLAS 性能的完整 GEMM kernel。